# NLP Exercise 1: Sentiment Analysis
---

## Loading necessary libraries.

In [1]:
%pip install nltk
%pip install scikit-learn
%pip install numpy
%pip install python-utils
%pip install gensim
%pip install torch

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [61]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from nltk.corpus import twitter_samples
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec
import numpy as np
import re
import nltk
import string
import pandas as pd
from nltk.corpus import twitter_samples, stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import TweetTokenizer

##  Traditional Sentiment Analysis approach


### Preprocessing the dataset



Tokenization after normalization

In [62]:
# Import nltk samples, stopwords
nltk.download('twitter_samples')
nltk.download('stopwords')

# Read corpus package
print(twitter_samples.fileids())

['negative_tweets.json', 'positive_tweets.json', 'tweets.20150430-223406.json']


[nltk_data] Downloading package twitter_samples to
[nltk_data]     C:\Users\minhn\AppData\Roaming\nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\minhn\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [63]:
def process_tweet(tweet):
    """
    Process tweet function.
    
    Input:
        tweet: a string containing a tweet.
    Output:
        tweets_clean: a list of words containing the processed tweet.

    """
    stemmer = PorterStemmer()
    stopwords_english = stopwords.words('english')
    # remove stock market tickers like $GE
    tweet = re.sub(r'\$\w*', '', tweet)
    # remove old style retweet text "RT"
    tweet = re.sub(r'^RT[\s]+', '', tweet)
    # remove hyperlinks
    tweet = re.sub(r'https?://[^\s\n\r]+', '', tweet)
    # remove hashtags
    # only removing the hash # sign from the word
    tweet = re.sub(r'#', '', tweet)
    # tokenize tweets
    tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True,
                               reduce_len=True)
    tweet_tokens = tokenizer.tokenize(tweet)

    tweets_clean = []
    for word in tweet_tokens:
        if ( word not in stopwords_english and  # remove stopwords
                word not in string.punctuation):  # remove punctuation
            stem_word = stemmer.stem(word)  # stemming word
            tweets_clean.append(stem_word)

    return tweets_clean

In [64]:
# Extract positive and negative tweets
positive_tweets = twitter_samples.strings('positive_tweets.json')
negative_tweets = twitter_samples.strings('negative_tweets.json')

all_tweets = positive_tweets + negative_tweets

# Create labels: 1 for positive, 0 for negative
positive_labels = [1] * len(positive_tweets)
negative_labels = [0] * len(negative_tweets)

all_labels = positive_labels + negative_labels


Create DataFrame for visualization

In [65]:
def create_dataframe(tweets, labels):
    """
    Create DataFrame to visualize the dataset.
    
    Input:
        tweet: a list containing tweet texts.
        labels: list containing labels for each tweet (1 for positve, 0 for negative).
    Output:
        A DataFrame with 2 columns of tweets and labels.

    """
    if len(tweets) != len(labels):
        raise ValueError

    df = pd.DataFrame({'tweets': tweets, 'labels': labels})

    return df

In [66]:
# Create DataFrame for tweets and labels
df = create_dataframe(all_tweets, all_labels)
print(df.head())

                                              tweets  labels
0  #FollowFriday @France_Inte @PKuchly57 @Milipol...       1
1  @Lamb2ja Hey James! How odd :/ Please call our...       1
2  @DespiteOfficial we had a listen last night :)...       1
3                               @97sides CONGRATS :)       1
4  yeaaaah yippppy!!!  my accnt verified rqst has...       1


In [67]:
# Preparing data for the Word2Vec model
cleaned_tweets = []
for tweet in all_tweets:
    cleaned_tweet = process_tweet(tweet)
    cleaned_tweets.append(cleaned_tweet)

In [68]:
# Word Embeddings using Word2Vec model
word2vec_model = Word2Vec(cleaned_tweets, vector_size=20,
                          window=5, min_count=5, workers=4)

word_embeddings = word2vec_model.wv

In [69]:
# Example using word embeddings
print(word_embeddings['listen'])

[ 0.14103504  0.16837461 -0.16333139  0.4435657  -0.23132119 -0.16949935
  0.05024713  0.8371735  -0.47512817  0.22605471  0.22104889 -0.04142627
  0.78461576 -0.48165962  0.54754716  0.45742437  0.8568998  -0.28989914
 -0.16861092 -0.43229347]


### Create tweet embeddings

Since each tweet consists of multiple words, convert each tweet into a single vector representation by averaging the Word2Vec embeddings of all the words in the tweet. This averaged vector will represent the tweet in a fixed-dimensional space, suitable for input into a classifier.

In [70]:
def get_tweet_embedding(tweet_tokens, word2vec_model):
    """
    Generate the embedding for a tweet by averaging word vectors.
    
    Input: 
        tweet_tokens: a list of tokens from processed tweet.
        word2vec_model: a trained Word2Vec model that contains word embeddings.
    Output:
        tweet_embedding: a numpy array representing the averaged embedding 
                        vector for a tweet. The dimension of the array is 
                        equal to the vector_size of the Word2Vec model.

    """
    tweet_vecs = []

    for word in tweet_tokens:
        if word in word2vec_model.wv:
            tweet_vecs.append(word2vec_model.wv[word])

    if len(tweet_vecs) == 0:
        return np.zeros(word2vec_model.vector_size)

    tweet_embedding = np.mean(tweet_vecs, axis=0)

    return tweet_embedding

In [71]:
# Example for using get_tweet_embeddings function
tweet1 = df['tweets'][4]
tweet1_cleaned = process_tweet(tweet1)
print(tweet1_cleaned)
tweet1_embedding = get_tweet_embedding(tweet1, word2vec_model)
print(tweet1_embedding)

['yeaaah', 'yipppi', 'accnt', 'verifi', 'rqst', 'succeed', 'got', 'blue', 'tick', 'mark', 'fb', 'profil', ':)', '15', 'day']
[ 0.15654765  0.10098389 -0.11507876  0.3247729  -0.15673473 -0.14418241
  0.03299037  0.58663213 -0.37352663  0.18336178  0.20032808 -0.06626335
  0.560945   -0.35189727  0.4108675   0.30635494  0.61493987 -0.15499353
 -0.12545854 -0.3423222 ]


Prepare and split the data into training and testing sets

In [72]:
# Convert labels to a numpy array
labels = np.array(all_labels)

# Generate embeddings for all tweets in the dataset
tweet_embeddings = np.array([get_tweet_embedding(
    tweet, word2vec_model) for tweet in cleaned_tweets])

# Split the dataset into training and test sets (80% train, 20% test)
x_train, x_test, y_train, y_test = train_test_split(
    tweet_embeddings, labels, test_size=0.2, random_state=42)


### Train Classifier

Train a Classifier using Logistic Regression as a baseline classifier, which works well for binary classification tasks like sentiment analysis

In [73]:
# Initialize and train the logistic regression classifier
clf = LogisticRegression(random_state=42)
clf.fit(x_train, y_train)

# Predict on the test set
y_pred = clf.predict(x_test)

### Evaluate the model

In [74]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy: ", accuracy)

# Detailed classification report
print(classification_report(y_test, y_pred))

Test Accuracy:  0.8645
              precision    recall  f1-score   support

           0       0.84      0.90      0.87       988
           1       0.89      0.83      0.86      1012

    accuracy                           0.86      2000
   macro avg       0.87      0.86      0.86      2000
weighted avg       0.87      0.86      0.86      2000



Example prediction by using the trained model

In [75]:
def predict_tweet(tweet, word2vec_model, clf):
    """
    Predict sentiment of a tweet using a trained Word2Vec model and classifier.
    
    Input:
        tweet: raw text to predict sentiment.
        word2vec_model: a trained Word2Vec model containing word embeddings.
        clf: a trained classifier for sentiment predictions.
    Output:
        Returns "Positive" if the predicted sentiment is positive, 
                otherwise returns "Negative".
                
    """
    processed_tweet = process_tweet(tweet)
    tweet_embedding = get_tweet_embedding(processed_tweet, word2vec_model)
    prediction = clf.predict([tweet_embedding])

    return "Positive" if prediction == 1 else "Negative"

In [76]:
# Example prediction
new_tweet = "I like to study NLP <3"
print("Sentiment: ", predict_tweet(new_tweet, word2vec_model, clf))

Sentiment:  Positive


## Multi-Layer Perceptrons

Multi-Layer Perceptrons (MLP) can enhance the predictive power of sentiment analysis model by allowing it to capture more complex patterns in the data.

### Outline for building a Multi-Layer Perceptrons for Sentiment Analysis

#### 1. Data Preparation: 
- Use the embeddings generated for each tweet as input features for the MLP.

#### 2. Model Architecture:
- Design a simple MLP with several fully connected layers (dense layers), an activation function (ReLu) for non-linearity, and dropout layers to prevent overfitting.

- Use a final ouput layer with a sigmoid activation function for binary classification.

#### 3. Training and Evaluation:
- Train the MLP on the training set, validate on the test set, and evaluate performance using accuracy and a classification report.

#### 4. Hyperparameter Tuning:
- Experiment with the number of layers, number of neurons, dropout rates, and learning rate to optimize performance.

### Import necessary libraries

In [77]:
import torch
import torch.nn as nn
import torch.optim as optim

### Implement the Multi-Layer Perceptron (MLP) class in PyTorch

In [78]:
class SentimentMLP(nn.Module):
    def __init__(self, embedding_dim: int, 
                 hidden_dim1: int, 
                 hidden_dim2: int, 
                 dropout: float):
        """
        Initialize the Multi-Layer Perceptrons (MLP) model for sentiment analysis.

        Parameters:
            input_dim (int): dimension of the input features (tweet embedding size).
            hidden_dim1 (int): Number of neurons in the first hidden layer.
            hidden_dim2 (int): Number of neurons in the second hidden layer.
            dropout_rate (float): Dropout rate to prevent overfitting.

        """
        super(SentimentMLP, self).__init__()

        # Using nn.Sequential to stack layers
        self.network = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim1),  # First hidden layer
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim1, hidden_dim2),  # Second hidden layer
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim2, 1),  # Output layer
            nn.Sigmoid()  # Sigmoid activation for binary classification
        )
    
    def forward(self, x):
        return self.network(x)

### Prepare the data

In [79]:
X = np.array(tweet_embeddings)  # tweet_embeddings generated from Word2Vec
y = np.array(labels)  # labels for the tweets (1 for positive, 0 for negative)

# Convert data to PyTorch tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

# Split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [80]:
# Initialize the parameters for the model
embedding_dim = X_train.shape[1]
hidden_dim1 = 64
hidden_dim2 = 32
dropout_rate = 0.3

Initialize the model

In [81]:
mlp_model = SentimentMLP(
    embedding_dim=embedding_dim,
    hidden_dim1=hidden_dim1,
    hidden_dim2=hidden_dim2,
    dropout=dropout_rate
)

Implementation for training stage

In [82]:
def train(model, x_train, y_train, criterion, optimizer, num_epochs, batch_size, print_every):
    """
    Train a model with the given dataset, loss function, and optimizer.

    Parameters:
        model: the neural network model to train.
        x_train: training features.
        y_train: training labels.
        criterion: loss function.
        optimizer: optimizer to update model parameters.
        num_epochs: number of epochs to train.
        batch_size: size of each batch to training.
        print_every: frequency of printing loss (e.g, every 5 epochs)
    
    Returns:
        A list of loss values for each epoch.
        
    """

    epoch_losses = []
    
    model.train()

    for epoch in range(num_epochs):
        # Shuffle the data at each epoch
        permutation = torch.randperm(x_train.size(0))
        epoch_loss = 0

        # Mini-batch training
        for i in range(0, x_train.size(0), batch_size):
            # Select mini-batch
            index = permutation[i:i+ batch_size]
            batch_x, batch_y = x_train[index], y_train[index]

            # Zero gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(batch_x).squeeze()
            loss = criterion(outputs, batch_y)

            # Backward pass and optimization
            loss.backward()
            optimizer.step()

            # Accumulate loss for each epoch
            epoch_loss += loss.item()
        
        # Average loss for each epoch
        avg_epoch_loss = epoch_loss / len(permutation)
        epoch_losses.append(avg_epoch_loss)

        # Print progress for each epoch
        if (epoch + 1) % print_every == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}, Loss: {avg_epoch_loss}]")
    
    return model, epoch_losses

Define the loss function and optimizer

In [83]:
criterion = nn.BCELoss()
optimizer = optim.Adam(mlp_model.parameters(), lr=0.001, weight_decay=1e-5)

Train model

In [84]:
model, losses = train(mlp_model, X_train, y_train, criterion, optimizer, num_epochs=20, batch_size=64, print_every=5)

Epoch [5/20, Loss: 0.006901019066572189]
Epoch [10/20, Loss: 0.005940233806148171]
Epoch [15/20, Loss: 0.004940275181084871]
Epoch [20/20, Loss: 0.004329608846455812]


Evaluate the model

In [85]:
def evaluate(model, x_test, y_test):
    """
    Evaluate the model on the test dataset and print performance metrics.
    
    Parameters:
        model (nn.Module): Trained model to evaluate.
        X_test (torch.Tensor): Test features.
        y_test (torch.Tensor): Test labels.
        
    Returns:
        Accuracy of the model on the test set.

    """

    model.eval()

    with torch.no_grad():
        outputs = model(x_test)
        predictions = (outputs > 0.5).int()

    accuracy = accuracy_score(y_test, predictions)

    print('Test Accuracy:', round(accuracy, 4))
    print(classification_report(y_test, predictions))

In [86]:
evaluate(model, X_test, y_test)

Test Accuracy: 0.904
              precision    recall  f1-score   support

         0.0       0.92      0.89      0.90       988
         1.0       0.89      0.92      0.91      1012

    accuracy                           0.90      2000
   macro avg       0.90      0.90      0.90      2000
weighted avg       0.90      0.90      0.90      2000



Predict Sentiment for new data

In [87]:
def preprocess_text(text):
    """
    Preprocessing the new data using process tweet function and Word2Vec model defined above.
    
    """
    processed_tokens = process_tweet(text)
    
    tweet_embedding = get_tweet_embedding(processed_tokens, word2vec_model)
    
    # Convert to a tensor and reshape to match the model's expected input shape
    tweet_embedding_tensor = torch.tensor(tweet_embedding, dtype=torch.float32).unsqueeze(0)
    return tweet_embedding_tensor

In [88]:
def predict_sentiment(model, text):
    """
    Predict the sentiment of a given text using the trained model.
    
    Parameters:
        model: the trained MLP model.
        text: input text to analyze.
    
    Returns:
        "Positive" if sentiment is positive, otherwise "Negative".
    """
    # Set the model to evaluation mode
    model.eval()
    
    # Preprocess the text and get the embedding
    input_tensor = preprocess_text(text)
    
    # Disable gradient computation for inference
    with torch.no_grad():
        output = model(input_tensor)  # Model outputs probability due to Sigmoid activation
    
    # Interpret the output
    prediction = (output.item() > 0.5)  # Threshold at 0.5 for binary classification
    sentiment = "Positive" if prediction else "Negative"
    
    return sentiment

In [89]:
# Example text to analyze
new_text = "Oh great, it's raining again!"

In [90]:
# Predict the sentiment
sentiment = predict_sentiment(model, new_text)
print(sentiment)

Positive


## Apply the same approach to Kaggle emotion dataset

`SentimentMLP` class and `train`, `evaluate`, `process_tweet` and `get_tweet_embedding` functions are reused.

### Prepare emotion data

In [91]:
emotion_df = pd.read_csv('combined_emotion.csv')
emotion_df = emotion_df[(emotion_df['emotion'] == 'joy') | (emotion_df['emotion'] == 'sad')]
# Vectorized assignment to avoid truth-value ambiguity for a Series
emotion_df['emotion_label'] = np.where(emotion_df['emotion'] == 'joy', 1, 0)
emotion_df.head()

,sentence,emotion,emotion_label
1,ive enjoyed being able to slouch about relax a...,sad,0
3,i dont know i feel so lost,sad,0
5,i was beginning to feel quite disheartened,sad,0
7,i fear that they won t ever feel that deliciou...,joy,1
9,i can still lose the weight without feeling de...,sad,0


In [92]:
all_sentences = emotion_df['sentence'].to_list()
all_labels_emotion = emotion_df['emotion_label'].to_list()

In [93]:
# Preparing data for the Word2Vec model
cleaned_sentences = []
for sentence in all_sentences:
    cleaned_sentence = process_tweet(sentence)
    cleaned_sentences.append(cleaned_sentence)

In [94]:
# Word Embeddings using Word2Vec model
word2vec_model_emotion = Word2Vec(cleaned_sentences, vector_size=20,
                          window=5, min_count=5, workers=4)

In [95]:
# Convert labels to a numpy array
labels_emotion = np.array(all_labels_emotion)

# Generate embeddings for all sentences in the dataset
sentence_embeddings = np.array([get_tweet_embedding(
    sentence, word2vec_model_emotion) for sentence in cleaned_sentences])

# Split the dataset into training and test sets (80% train, 20% test)
x_train_emotion, x_test_emotion, y_train_emotion, y_test_emotion = train_test_split(
    sentence_embeddings, labels_emotion, test_size=0.2, random_state=42)

### Train a Classifier using Logistic Regression

In [96]:
# Initialize and train the logistic regression classifier
clf_emotion = LogisticRegression(random_state=42)
clf_emotion.fit(x_train_emotion, y_train_emotion)

# Predict on the test set
y_pred_emotion = clf_emotion.predict(x_test_emotion)

In [97]:
# Calculate accuracy
accuracy = accuracy_score(y_test_emotion, y_pred_emotion)
print("Test Accuracy: ", round(accuracy, 4))

# Detailed classification report
print(classification_report(y_test_emotion, y_pred_emotion))

Test Accuracy:  0.8675
              precision    recall  f1-score   support

           0       0.86      0.85      0.85     24139
           1       0.87      0.89      0.88     28712

    accuracy                           0.87     52851
   macro avg       0.87      0.87      0.87     52851
weighted avg       0.87      0.87      0.87     52851



In [98]:
def predict_sentence_emotion(sentence, word2vec_model_emotion, clf):
    """
    Predict sentiment of a sentence using a trained Word2Vec model and classifier.
    
    Input:
        sentence: raw text to predict sentiment.
        word2vec_model: a trained Word2Vec model containing word embeddings.
        clf: a trained classifier for sentiment predictions.
    Output:
        Returns "Joy" if the predicted sentiment is joy, 
                otherwise returns "Sadness".
                
    """
    processed_sentence = process_tweet(sentence)
    sentence_embedding = get_tweet_embedding(processed_sentence, word2vec_model_emotion)
    prediction = clf.predict([sentence_embedding])

    return "Joy" if prediction == 1 else "Sadness"

In [99]:
# Example prediction
new_sentence = "i am excited that the summer holiday will begin soon"
print("Sentiment: ", predict_sentence_emotion(new_sentence, word2vec_model_emotion, clf_emotion))
new_sentence_2 = "i am hopeless about my future"
print("Sentiment: ", predict_sentence_emotion(new_sentence_2, word2vec_model_emotion, clf_emotion))

Sentiment:  Joy
Sentiment:  Sadness


### Train a similar Classifier using MLP

In [100]:
X_emotion = np.array(sentence_embeddings)  # sentence_embeddings generated from Word2Vec
y_emotion = np.array(labels_emotion)  # labels for the emotion (1 for joy, 0 for sad)

# Convert data to PyTorch tensors
X_emotion = torch.tensor(X_emotion, dtype=torch.float32)
y_emotion = torch.tensor(y_emotion, dtype=torch.float32)

# Split the data into training and test datasets
X_train_emotion, X_test_emotion, y_train_emotion_2, y_test_emotion_2 = train_test_split(X_emotion, y_emotion, test_size=0.2, random_state=42)

In [101]:
# Initialize the parameters for the model
embedding_dim = X_train_emotion.shape[1]
hidden_dim1 = 128
hidden_dim2 = 64
dropout_rate = 0.3

In [102]:
mlp_model_emotion = SentimentMLP(
    embedding_dim=embedding_dim,
    hidden_dim1=hidden_dim1,
    hidden_dim2=hidden_dim2,
    dropout=dropout_rate
)

In [103]:
criterion_emotion = nn.BCELoss()
optimizer_emotion = optim.Adam(mlp_model_emotion.parameters(), lr=0.001, weight_decay=1e-5)

In [104]:
model_emotion, losses_emotion = train(mlp_model_emotion, X_train_emotion, y_train_emotion_2, criterion_emotion, optimizer_emotion, num_epochs=50, batch_size=128, print_every=5)

Epoch [5/50, Loss: 0.0021507073445726655]
Epoch [10/50, Loss: 0.0021072535610710767]
Epoch [15/50, Loss: 0.0020842574152329096]
Epoch [20/50, Loss: 0.002065598425113252]
Epoch [25/50, Loss: 0.0020490456483604584]
Epoch [30/50, Loss: 0.0020416744593752313]
Epoch [35/50, Loss: 0.0020393730791627963]
Epoch [40/50, Loss: 0.002045979965941722]
Epoch [45/50, Loss: 0.0020252612265628236]
Epoch [50/50, Loss: 0.0020231154339755927]


In [105]:
evaluate(model_emotion, X_test_emotion, y_test_emotion_2)

Test Accuracy: 0.8936
              precision    recall  f1-score   support

         0.0       0.88      0.89      0.88     24139
         1.0       0.91      0.90      0.90     28712

    accuracy                           0.89     52851
   macro avg       0.89      0.89      0.89     52851
weighted avg       0.89      0.89      0.89     52851



In [106]:
def preprocess_text(text):
    """
    Preprocessing the new data using process tweet function and Word2Vec model defined above.
    
    """
    processed_tokens = process_tweet(text)
    
    tweet_embedding = get_tweet_embedding(processed_tokens, word2vec_model_emotion)
    
    # Convert to a tensor and reshape to match the model's expected input shape
    tweet_embedding_tensor = torch.tensor(tweet_embedding, dtype=torch.float32).unsqueeze(0)
    return tweet_embedding_tensor

In [107]:
def predict_sentiment(model, text):
    """
    Predict the sentiment of a given text using the trained model.
    
    Parameters:
        model: the trained MLP model.
        text: input text to analyze.
    
    Returns:
        "Joy" if sentiment is positive, otherwise "Sadness".
    """
    # Set the model to evaluation mode
    model.eval()
    
    # Preprocess the text and get the embedding
    input_tensor = preprocess_text(text)
    
    # Disable gradient computation for inference
    with torch.no_grad():
        output = model(input_tensor)  # Model outputs probability due to Sigmoid activation
    
    # Interpret the output
    prediction = (output.item() > 0.5)  # Threshold at 0.5 for binary classification
    sentiment = "Joy" if prediction else "Sadness"
    
    return sentiment

In [108]:
# Example text to analyze
new_sentence = "i am excited that the summer holiday will begin soon"
sentiment = predict_sentiment(model_emotion, new_sentence)
print("Sentiment:", sentiment)
new_sentence_2 = "i am hopeless about my future"
sentiment = predict_sentiment(model_emotion, new_sentence_2)
print("Sentiment:", sentiment)

Sentiment: Joy
Sentiment: Sadness


## Test emotion model with twitter dataset (only MLP model)

In [109]:
evaluate(model_emotion, X_test, y_test)

Test Accuracy: 0.494
              precision    recall  f1-score   support

         0.0       0.49      1.00      0.66       988
         1.0       0.50      0.00      0.00      1012

    accuracy                           0.49      2000
   macro avg       0.50      0.50      0.33      2000
weighted avg       0.50      0.49      0.33      2000



## Test twitter model with emotion dataset (only MLP model)

In [110]:
evaluate(model, X_test_emotion, y_test_emotion_2)

Test Accuracy: 0.4729
              precision    recall  f1-score   support

         0.0       0.46      0.99      0.63     24139
         1.0       0.86      0.04      0.07     28712

    accuracy                           0.47     52851
   macro avg       0.66      0.51      0.35     52851
weighted avg       0.68      0.47      0.33     52851

